In [73]:
import numpy as np
import random
import time
import json
from datetime import datetime
import pennylane as qp
import networkx as nx
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.linalg import expm

In [81]:
k = L = 2              # degree of the modeled polynomial function
DATASET_SIZE = 100                 # size of the full dataset
num_nodes = 8

TEST_SIZE=0.2
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [82]:
config = {
    "experiment_name": "classical",
    "num_nodes": num_nodes,
    "num_layers": L,
    "dataset_size": DATASET_SIZE,
    "test_size": TEST_SIZE,
    "seed": SEED
}

In [83]:
def get_encoding_matrix(adj_matrix):
    """
    Returns the Ising Hamiltonian for a given adjacency matrix, as defined in the project instructions.
    If the graph has no edges, returns the zero Hamiltonian.
    """
    n = adj_matrix.shape[0]
    coeffs = []
    obs = []
    for i in range(n):
        for j in range(i + 1, n):
            if adj_matrix[i, j] == 1:
                coeffs.append(1.)
                obs.append(qp.PauliZ(i) @ qp.PauliZ(j))
    if len(obs) == 0:
        return np.eye(2**adj_matrix.shape[0], dtype=complex)
    H_dense = qp.matrix(qp.Hamiltonian(coeffs, obs), wire_order=range(adj_matrix.shape[0]))
    return expm(1j * H_dense)

### Data Generation

In [84]:
dataset = []

threshold = np.log(num_nodes) / num_nodes   # The theoretical threshold for a graph to be connected is ln(n)/n
edge_probabilities = [random.uniform(threshold*0.9, threshold*1.5) for _ in range(DATASET_SIZE)]

connected_count = 0
disconnected_count = 0
for p in edge_probabilities:
    G = nx.erdos_renyi_graph(n=num_nodes, p=p)
    A = nx.to_numpy_array(G)
    H_G = get_encoding_matrix(A)
    if nx.is_connected(G):
        y = 0
        connected_count+=1
    else:
        y = 1
        disconnected_count+=1
    dataset.append((A, H_G, y))
config["connected_ratio"] = connected_count / len(dataset)

print(f"The full dataset contains {connected_count} connected graphs and {disconnected_count} disconnected graphs.")
train_data, test_data = train_test_split(dataset, test_size=TEST_SIZE, random_state=SEED)

The full dataset contains 50 connected graphs and 50 disconnected graphs.


In [85]:
def filter_isomorphisms(source_dataset):
    """
    Filters a dataset to contain only non-isomorphic graphs.
    If reference_graphs is provided, it also ensures no graph in the
    source_dataset is isomorphic to any graph in the reference_graphs.
    """

    unique_graphs = list()
    filtered_dataset = []

    for data_point in source_dataset:
        A, H_G, y = data_point
        G_current = nx.from_numpy_array(A)

        is_iso = False
        for G_seen in unique_graphs:
            if nx.is_isomorphic(G_current, G_seen):
                is_iso = True
                break

        if not is_iso:
            unique_graphs.append(G_current)
            filtered_dataset.append(data_point)

    return filtered_dataset

print(f"Original train size: {len(train_data)}")
train_data_clean = filter_isomorphisms(train_data)
print(f"Cleaned train size (no internal isomorphisms): {len(train_data_clean)}\n")

train_data = train_data_clean
#X_train = np.array([data_point[0] for data_point in train_data])
X_train = np.array([np.concatenate([data_point[1].flatten().real,data_point[1].flatten().imag])
    for data_point in train_data])
y_train = np.array([data_point[2] for data_point in train_data])
#X_test = np.array([data_point[0] for data_point in test_data])
X_test = np.array([np.concatenate([data_point[1].flatten().real,data_point[1].flatten().imag])
    for data_point in test_data])
y_test = np.array([data_point[2] for data_point in test_data])

Original train size: 80
Cleaned train size (no internal isomorphisms): 75



Classical Polynomial Model

In [86]:
# - PolynomialFeatures generates all combinations of features up to degree k
# - LogisticRegression applies the free weights (w) and a sigmoid for binary classification
classical_model = make_pipeline(
    PolynomialFeatures(degree=L, include_bias=False),
    # penalty=None ensures the coefficients w are "completely free"
    LogisticRegression(fit_intercept=False, penalty=None, max_iter=1000)
)

### Training

In [80]:
print("Training the classical model...")

start_time = time.time()

classical_model.fit(X_train, y_train)

end_time = time.time()
duration = end_time - start_time
print(duration)

Training the classical model...


MemoryError: Unable to allocate 1.25 PiB for an array with shape (80, 2199026401280) and data type float64

### Performance Evaluation

In [70]:
train_acc = accuracy_score(y_train, classical_model.predict(X_train))
test_acc = accuracy_score(y_test, classical_model.predict(X_test))

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy:  {test_acc:.4f}")

Training Accuracy: 1.0000
Testing Accuracy:  0.6500


In [71]:
log_reg_model = classical_model.named_steps['logisticregression']
num_params = log_reg_model.coef_.size

print(f"The model has {num_params} free parameters.")

The model has 2100224 free parameters.


In [72]:
results = {
    **config,
    "num_params": num_params,
    "final_train_acc": round(float(train_acc), 3),
    "final_test_acc": round(float(test_acc), 3),
    "total_training_time_sec": round(duration, 2),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open("experiments_log.jsonl", "a") as f:
    f.write(json.dumps(results) + "\n")